# MEG Stroke Intervention - Data Exploration

This notebook explores the synthetic MEG data generated for training the stroke intervention model.

- Channel signal visualization across conditions
- Power spectral density analysis (mu and beta bands)
- Label distributions per stroke type
- Laterality index analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load synthetic data
data_dir = Path('..') / 'data' / 'synthetic'
train = np.load(data_dir / 'train.npz')
metadata = np.load(data_dir / 'metadata.npz', allow_pickle=True)

X = train['data']           # (N, 6, 100)
y = train['labels']         # (N, 3)
conditions = train['conditions']  # (N,)

channel_names = list(metadata['channel_names'])
condition_names = list(metadata['condition_names'])

print(f'Data shape: {X.shape}')
print(f'Labels shape: {y.shape}')
print(f'Channels: {channel_names}')
print(f'Conditions: {condition_names}')
print(f'Condition counts: {np.bincount(conditions).tolist()}')

In [ ]:
# Plot example signals for each condition
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for cond_idx, (ax, cond_name) in enumerate(zip(axes, condition_names)):
    mask = conditions == cond_idx
    sample_idx = np.where(mask)[0][0]
    
    t = np.arange(100) / 200.0 * 1000  # ms
    for ch in range(6):
        ax.plot(t, X[sample_idx, ch], label=channel_names[ch], alpha=0.7)
    
    ax.set_title(f'{cond_name.upper()} - Example Signal', fontsize=12)
    ax.set_ylabel('Amplitude')
    ax.legend(loc='upper right', fontsize=8, ncol=3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

In [ ]:
# Label distributions per condition
label_names = ['valve_extension', 'force_magnitude', 'trigger_delay']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#2196F3', '#F44336', '#FF9800']

for i, (ax, lname) in enumerate(zip(axes, label_names)):
    for cond_idx, cond_name in enumerate(condition_names):
        mask = conditions == cond_idx
        ax.hist(y[mask, i], bins=30, alpha=0.5, label=cond_name, density=True)
    ax.set_title(lname)
    ax.set_xlabel('Value')
    ax.legend(fontsize=8)

plt.suptitle('Label Distributions by Condition', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Power Spectral Density analysis
from scipy import signal as scipy_signal

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for cond_idx, (ax, cond_name) in enumerate(zip(axes, condition_names)):
    mask = conditions == cond_idx
    # Average PSD across samples for each channel
    for ch in range(6):
        freqs, psd = scipy_signal.welch(X[mask, ch, :], fs=200, nperseg=64, axis=-1)
        mean_psd = psd.mean(axis=0)
        ax.semilogy(freqs, mean_psd, label=channel_names[ch], alpha=0.8)
    
    # Mark mu and beta bands
    ax.axvspan(8, 12, alpha=0.1, color='green', label='Mu (8-12Hz)')
    ax.axvspan(12, 32, alpha=0.1, color='blue', label='Beta (12-32Hz)')
    ax.set_title(f'{cond_name.upper()} - Average PSD', fontsize=12)
    ax.set_ylabel('Power')
    ax.legend(loc='upper right', fontsize=7, ncol=4)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

In [ ]:
# Laterality analysis: compare left vs right hemisphere power
left_channels = [0, 2, 4]   # C3, FC3, CP3
right_channels = [1, 3, 5]  # C4, FC4, CP4

laterality_indices = []

for i in range(len(X)):
    left_power = np.mean(X[i, left_channels, :] ** 2)
    right_power = np.mean(X[i, right_channels, :] ** 2)
    denom = left_power + right_power
    li = (right_power - left_power) / denom if denom > 1e-10 else 0.0
    laterality_indices.append(li)

laterality_indices = np.array(laterality_indices)

fig, ax = plt.subplots(figsize=(10, 5))
for cond_idx, cond_name in enumerate(condition_names):
    mask = conditions == cond_idx
    ax.hist(laterality_indices[mask], bins=40, alpha=0.5, label=cond_name, density=True)

ax.axvline(0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Laterality Index (R-L)/(R+L)')
ax.set_ylabel('Density')
ax.set_title('Hemispheric Laterality Index Distribution')
ax.legend()
plt.tight_layout()
plt.show()

print('Mean laterality by condition:')
for cond_idx, cond_name in enumerate(condition_names):
    mask = conditions == cond_idx
    print(f'  {cond_name}: {laterality_indices[mask].mean():.4f} +/- {laterality_indices[mask].std():.4f}')